In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("/content/project_data_cleaned_v1.csv")
data.head()

,comment_id,text,published_at,roberta_neg,roberta_neu,roberta_pos,roberta_label,num_words
0,UgwoiJ6zDlEHgNHbKHl4AaABAg,mike is the most traumatic in this season his ...,2025-12-27T02:17:44Z,0.971900,0.025935,0.002164,negative,42
1,UgyXJRcSbn74ajhkmBN4AaABAg,"this was so bad yall. wth was that writing, th...",2025-12-27T02:13:56Z,0.975026,0.022050,0.002924,negative,54
2,UgzhXrqNFCkMLMOB6dh4AaABAg,not a single shot from final episode,2025-12-27T01:51:33Z,0.390147,0.579394,0.030459,neutral,7
3,UgxQPzUbRC8ECcXroFF4AaABAg,rated j for jesus: a box office blasphemy – ho...,2025-12-27T01:35:21Z,0.299472,0.592630,0.107898,neutral,15
4,UgzQIyIMuD3WZha30dd4AaABAg,does hollyweird ever stop attacking christiani...,2025-12-27T01:30:37Z,0.972716,0.024830,0.002454,negative,27


In [3]:
data = data.loc[:,["text", "roberta_label"]].rename(columns={"roberta_label" : "label"})

In [4]:
data.shape

(19596, 2)

In [6]:
# preprocessing text
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Ensure NLTK data is downloaded
nltk.download('stopwords')
nltk.download('wordnet')

# Preprocessing function
def preprocess_comment(comment):
    comment = comment.lower().strip()
    comment = re.sub(r'\n', ' ', comment)
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])
    return comment

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [8]:
data["text"] = data["text"].apply(preprocess_comment)

In [10]:
def encode_label(string):
  """takes a sentiment label and converts it to 0 if neutral, 1 if positive and -1 if negative"""
  if string == "positive":
    return 1
  elif string == "negative":
    return -1
  else:
    return 0

data["label"] = data["label"].apply(encode_label)

In [11]:
data.sample(10)

,text,label
3613,two type audience comment section 1.who threat...,-1
3080,volume 2?!are telling year waiting release fin...,0
14818,let gooo,1
3278,120 actually insaneeeeee,1
9037,max better make alive istg.,0
5591,cant wait,1
16861,please save steve dustin,0
7619,"nooo, another trailer without confirming billy",-1
16678,134 let goo shock jock theory debunkedit fores...,0
2123,like comment byler button lol,0


In [13]:
data.to_csv("data_for_model_building.csv", index=False)

In [14]:
#  splitting data into X and y
X = data.loc[:, "text"]
y = data.loc[:, "label"]

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Vectorizing using Tfidf

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [17]:
# TF-IDF vectorizer with max 10,000 features (fit only on X_train)
vectorizer_tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = vectorizer_tfidf.fit_transform(X_train)
X_test_tfidf = vectorizer_tfidf.transform(X_test)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm

In [24]:
# Define classifiers
classifiers = {
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200)
}

# Evaluate classifiers with tqdm
results = []

for name, clf in tqdm(classifiers.items(), desc="Evaluating classifiers"):
    clf.fit(X_train_tfidf, y_train)
    y_pred = clf.predict(X_test_tfidf)
    results.append({
        "Model": name,
        "Precision (macro)": precision_score(y_test, y_pred, average='macro', zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average='macro', zero_division=0),
        "F1 Score (macro)": f1_score(y_test, y_pred, average='macro', zero_division=0)
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)
print(results_df)

# Optional: map predictions back to original labels
# y_pred_original = le.inverse_transform(y_pred)

Evaluating classifiers: 100%|██████████| 2/2 [02:03<00:00, 61.83s/it]

           Model  Precision (macro)  Recall (macro)  F1 Score (macro)
0    Naive Bayes           0.666360        0.638870          0.642179
1  Random Forest           0.672807        0.676288          0.674014


In [26]:
results_df

,Model,Precision (macro),Recall (macro),F1 Score (macro)
0,Naive Bayes,0.666360,0.638870,0.642179
1,Random Forest,0.672807,0.676288,0.674014


In [28]:
# Encode labels to 0,1,2 for XGBoost and LightGBM
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test) # -1->0, 0->1, 1->2

In [31]:
# Only XGBoost and LightGBM
classifiers = {
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42)
}

# Evaluate classifiers with tqdm
results = []

for name, clf in tqdm(classifiers.items(), desc="Evaluating LightGBM & XGBoost"):
    clf.fit(X_train_tfidf, y_train_encoded)
    y_pred = clf.predict(X_test_tfidf)
    results.append({
        "Model": name,
        "Precision (macro)": precision_score(y_test_encoded, y_pred, average='macro', zero_division=0),
        "Recall (macro)": recall_score(y_test_encoded, y_pred, average='macro', zero_division=0),
        "F1 Score (macro)": f1_score(y_test_encoded, y_pred, average='macro', zero_division=0)
    })

# Separate results DataFrame
results_df_lightgbm_xgb = pd.DataFrame(results)
results_df_lightgbm_xgb

Evaluating LightGBM & XGBoost:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [22:51:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
Evaluating LightGBM & XGBoost:  50%|█████     | 1/2 [00:18<00:18, 18.27s/it]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.110743 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 19135
[LightGBM] [Info] Number of data points in the train set: 15676, number of used features: 639
[LightGBM] [Info] Start training from score -0.960038
[LightGBM] [Info] Start training from score -1.051573
[LightGBM] [Info] Start training from score -1.317761


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Evaluating LightGBM & XGBoost: 100%|██████████| 2/2 [00:21<00:00, 10.92s/it]


,Model,Precision (macro),Recall (macro),F1 Score (macro)
0,XGBoost,0.715323,0.678288,0.686064
1,LightGBM,0.697441,0.671406,0.678072


- Out of Multinomial Naive bayes, Random Forest, xgboost and lightgbm, xgboost seems to perform best in terms of all given metrics.

### Hyperparamter tuning on Xgboost, random forest and light gbm

In [33]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 8.0 MB/s eta 0:00:00


In [35]:
# Random forest hyperparameter tuning
import optuna


def objective_rf(trial):
    # Hyperparameter search space
    n_estimators = trial.suggest_int('n_estimators', 50, 500)
    max_depth = trial.suggest_int('max_depth', 2, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])


    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    clf.fit(X_train_tfidf, y_train_encoded)
    y_pred = clf.predict(X_test_tfidf)
    f1 = f1_score(y_test_encoded, y_pred, average='macro')
    return f1

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=50)

print("Best Random Forest params:", study_rf.best_params)
print("Best Random Forest F1 (macro):", study_rf.best_value)

[I 2025-12-30 22:58:32,643] A new study created in memory with name: no-name-db751d7e-5a39-418a-b9af-9a7d14f8b97d
[I 2025-12-30 22:58:36,352] Trial 0 finished with value: 0.18298476155222407 and parameters: {'n_estimators': 397, 'max_depth': 12, 'min_samples_split': 3, 'min_samples_leaf': 13, 'max_features': 'log2'}. Best is trial 0 with value: 0.18298476155222407.
[I 2025-12-30 22:58:37,585] Trial 1 finished with value: 0.18298476155222407 and parameters: {'n_estimators': 244, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 0 with value: 0.18298476155222407.
[I 2025-12-30 23:01:00,705] Trial 2 finished with value: 0.6142472520048304 and parameters: {'n_estimators': 421, 'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 16, 'max_features': None}. Best is trial 2 with value: 0.6142472520048304.
[I 2025-12-30 23:01:01,811] Trial 3 finished with value: 0.18298476155222407 and parameters: {'n_estimators': 464, 'max_depth': 2

Best Random Forest params: {'n_estimators': 170, 'max_depth': 30, 'min_samples_split': 9, 'min_samples_leaf': 1, 'max_features': None}
Best Random Forest F1 (macro): 0.631563804600075


In [36]:
# for random forest
best_params_rf = study_rf.best_params
best_f1_rf = study_rf.best_value

In [37]:
# running random forest with the best parameter values
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

best_rf = RandomForestClassifier(
    **best_params_rf,
    random_state=42
)

best_rf.fit(X_train_tfidf, y_train)

RandomForestClassifier(max_depth=30, max_features=None, min_samples_split=9,
                       n_estimators=170, random_state=42)

In [38]:
y_pred = best_rf.predict(X_test_tfidf)

rf_results = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "F1 Score (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Best Params": best_params_rf
}


In [39]:
results_df_rf = pd.DataFrame([rf_results])
results_df_rf

,Model,Accuracy,Precision (macro),Recall (macro),F1 Score (macro),Best Params
0,Random Forest,0.649235,0.704407,0.626505,0.631564,"{'n_estimators': 170, 'max_depth': 30, 'min_sa..."


### Xgboost hyperparamter tuning

In [40]:
from xgboost import XGBClassifier

def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'use_label_encoder': False,
        'eval_metric': 'mlogloss',
        'random_state': 42
    }
    clf = XGBClassifier(**params)
    clf.fit(X_train_tfidf, y_train_encoded)
    y_pred = clf.predict(X_test_tfidf)
    f1 = f1_score(y_test_encoded, y_pred, average='macro')
    return f1

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=35)

print("Best XGBoost params:", study_xgb.best_params)
print("Best XGBoost F1 (macro):", study_xgb.best_value)

[I 2025-12-30 23:51:47,823] A new study created in memory with name: no-name-c97ffc9f-ef0e-458b-ace2-f1d68fdf0765
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [23:51:48] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2025-12-30 23:54:41,901] Trial 0 finished with value: 0.68961074725505 and parameters: {'n_estimators': 439, 'max_depth': 15, 'learning_rate': 0.022098697493506077, 'subsample': 0.6414068539744321, 'colsample_bytree': 0.7715881535448148, 'gamma': 1.0670051264780767}. Best is trial 0 with value: 0.68961074725505.
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [23:54:41] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2025-12-30 23:55:08,584] Trial 1 finished with value: 0.6839428669570365 and parameters: {'n_estimators': 149, 'max_

Best XGBoost params: {'n_estimators': 384, 'max_depth': 9, 'learning_rate': 0.14432103031505297, 'subsample': 0.9636819545620402, 'colsample_bytree': 0.7665904431247078, 'gamma': 1.77936321036146}
Best XGBoost F1 (macro): 0.6990078056336602


In [42]:
# retrieving best parameters for XgBoost
best_params_xgb = study_xgb.best_params
best_f1_xgb = study_xgb.best_value

In [43]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

best_xgb = XGBClassifier(
    **best_params_xgb,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

best_xgb.fit(X_train_tfidf, y_train_encoded)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [00:33:44] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7665904431247078, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=1.77936321036146, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.14432103031505297,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=384, n_jobs=None,
              num_parallel_tree=None, ...)

In [44]:
y_pred_xgb = best_xgb.predict(X_test_tfidf)

xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test_encoded, y_pred_xgb),
    "Precision (macro)": precision_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "F1 Score (macro)": f1_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "Best Params": best_params_xgb
}

In [45]:
results_df_xgb = pd.DataFrame([xgb_results])
results_df_xgb

,Model,Accuracy,Precision (macro),Recall (macro),F1 Score (macro),Best Params
0,XGBoost,0.703571,0.719762,0.691895,0.699008,"{'n_estimators': 384, 'max_depth': 9, 'learnin..."


## Improving XgBoost

### finding an optimal value of max_features for Tfidf

In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

In [50]:
def objective_tfidf(trial):
    max_features = trial.suggest_int("max_features", 2000, 50000, step=2000)

    tfidf = TfidfVectorizer(
        max_features=max_features,
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9
    )

    X_train_vec = tfidf.fit_transform(X_train)
    X_test_vec = tfidf.transform(X_test)

    clf = LogisticRegression(
        max_iter=1000,
        n_jobs=-1,
        multi_class="auto"
    )

    clf.fit(X_train_vec, y_train_encoded)
    y_pred = clf.predict(X_test_vec)

    return f1_score(y_test_encoded, y_pred, average="macro")

In [51]:
import optuna

study_tfidf = optuna.create_study(direction="maximize")
study_tfidf.optimize(objective_tfidf, n_trials=25)

print("Best TF-IDF max_features:", study_tfidf.best_params)
print("Best macro F1:", study_tfidf.best_value)

[I 2025-12-31 00:47:12,922] A new study created in memory with name: no-name-a03f90cb-5e0d-4f1d-88ce-d1a8e3fcdbfc
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
[I 2025-12-31 00:47:17,955] Trial 0 finished with value: 0.7033764150115435 and parameters: {'max_features': 14000}. Best is trial 0 with value: 0.7033764150115435.
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
[I 2025-12-31 00:47:21,375] Trial 1 finished with value: 0.7033764150115435 and parameters: {'max_features': 44000}. Best is trial 0 with value: 0.7033

Best TF-IDF max_features: {'max_features': 14000}
Best macro F1: 0.7033764150115435


In [52]:
tfidf_results = {
    "best_max_features": study_tfidf.best_params["max_features"],
    "best_macro_f1": study_tfidf.best_value
}

In [70]:
vectorizer2 = TfidfVectorizer(max_features=30000)
X_train_tfidf2 = vectorizer2.fit_transform(X_train)
X_test_tfidf2 = vectorizer2.transform(X_test)

In [71]:
# training XgBoost with the best params and the most optimal max_features values
best_xgb = XGBClassifier(
    **best_params_xgb,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)

best_xgb.fit(X_train_tfidf2, y_train_encoded)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [01:07:53] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7665904431247078, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=1.77936321036146, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.14432103031505297,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=384, n_jobs=None,
              num_parallel_tree=None, ...)

In [72]:
y_pred_xgb = best_xgb.predict(X_test_tfidf2)

xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test_encoded, y_pred_xgb),
    "Precision (macro)": precision_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "F1 Score (macro)": f1_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "Best Params": best_params_xgb
}

In [73]:
results_df_xgb2 = pd.DataFrame([xgb_results])
results_df_xgb2

,Model,Accuracy,Precision (macro),Recall (macro),F1 Score (macro),Best Params
0,XGBoost,0.703571,0.719762,0.691895,0.699008,"{'n_estimators': 384, 'max_depth': 9, 'learnin..."


In [75]:
results_df_xgb["Best Params"].values

array([{'n_estimators': 384, 'max_depth': 9, 'learning_rate': 0.14432103031505297, 'subsample': 0.9636819545620402, 'colsample_bytree': 0.7665904431247078, 'gamma': 1.77936321036146}],
      dtype=object)

In [76]:
# tfidf with max_features=14000
vectorizer3 = TfidfVectorizer(max_features=14000)
X_train2 = vectorizer3.fit_transform(X_train)
X_test2 = vectorizer3.transform(X_test)

In [78]:
xgboost3 = xgb_final = XGBClassifier(
    n_estimators=384,
    max_depth=9,
    learning_rate=0.14432103031505297,
    subsample=0.9636819545620402,
    colsample_bytree=0.7665904431247078,
    gamma=1.77936321036146,
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgboost3.fit(X_train2, y_train_encoded)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.7665904431247078, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='mlogloss', feature_types=None, feature_weights=None,
              gamma=1.77936321036146, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.14432103031505297,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=384, n_jobs=-1, num_class=3, ...)

In [79]:
y_pred_xgb = xgboost3.predict(X_test2)

xgb_results = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test_encoded, y_pred_xgb),
    "Precision (macro)": precision_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "Recall (macro)": recall_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "F1 Score (macro)": f1_score(y_test_encoded, y_pred_xgb, average="macro", zero_division=0),
    "Best Params": best_params_xgb
}

In [80]:
results_df_xgb2 = pd.DataFrame([xgb_results])
results_df_xgb2

,Model,Accuracy,Precision (macro),Recall (macro),F1 Score (macro),Best Params
0,XGBoost,0.703571,0.719762,0.691895,0.699008,"{'n_estimators': 384, 'max_depth': 9, 'learnin..."
